In [1]:
%pip install langgraph langchain_openai langchain_community langchain_anthropic
%pip install tavily-python
%pip install ipython
%pip install pygraphviz
%pip install python-dotenv
%pip install langchain-anthropic 
%pip install sentence_transformers elasticsearch cohere
%pip install asyncio aiohttp


[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Using cached pygraphviz-1.13.tar.gz (104 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for pygraphviz (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [63 lines of output]
      running bdist_wheel
      running build
      running build_py
      creating build
      creating build/lib.macosx-14.0-arm6

In [5]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [24]:
from langchain.globals import set_llm_cache

from tavily import TavilyClient

set_llm_cache(None)

tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

results = tavily.search(query="What is a few-shot prompting?", max_results=3)

[r["content"] for r in results["results"]]

['Few-shot prompting is a technique in which an AI model is given a few examples of a task to learn from before generating a response, using those examples to improve its performance on similar tasks. Large language models can understand and write text that sounds very human-like. But when it comes to getting these models to do this in the exact ...',
 "How does Few-Shot Prompting Work\nWhat is Few-Shot Prompting?\nFew-shot prompting is a technique where you provide a machine learning model, particularly a language model, with a small set of examples to guide its behavior for a specific task. On This Page\nUnderstanding Few-Shot Prompting in Prompt Engineering\nPublished on 12/17/2023\nIntroduction to Few-Shot Prompting\nWelcome to the fascinating world of Few-Shot Prompting in Prompt Engineering! In this example, we'll use few-shot prompting with intermediate steps to determine who lived longer: Muhammad Ali or Alan Turing.\nSteps to Follow:\nDefine the Task: The end goal is to find o

# Design Resarch Agent 

![](./images/Research%20Agent.png)


In [264]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import Annotated, List, Optional
from langchain_core.tools import tool

sub_topic_generator_prompt = ChatPromptTemplate.from_template(
    """ 
    You are an expert curriculum designer for beginner developers. Please generate 4-5 crucial and fundamental subtopics for a specific main topic, considering the course name and overall roadmap provided below. These subtopics should cover the most essential content of the main topic and include core concepts and skills that beginner developers must know.

    Course Name: {course_name}

    Complete Course Roadmap:
    {roadmap}

    Main Topic to Focus On: {main_topic}

    Target Audience: Beginner Developers
    - Have basic programming knowledge but limited real-world development experience
    - Need to acquire the most important and essential knowledge about the topic
    - Should focus on fundamental concepts that require a deep understanding

    Generated subtopics should meet the following criteria:
    1. Address the most crucial and fundamental concepts of the main topic
    2. Be essential for the long-term growth of beginner developers
    3. Include knowledge or skills absolutely necessary in real-world development
    4. Establish a strong foundation in the topic while suggesting possibilities for advanced topics
    5. Clearly indicate what beginner developers will be able to do after learning
    6. Include content that corrects common misconceptions or mistakes about the topic
    7. Introduce industry standards or best practices when possible
    8. Have clear connections between subtopics and enable a comprehensive understanding of the main topic
    9. Align with the context of the overall course roadmap and consider connections with other topics

    Please generate subtopics that meet the above criteria and output the results in the following JSON format:

    {{
    "main_topic": "Name of the Main Topic",
    "sub_topics": [
        {{
            "title": "Subtopic 1 Title",
            "importance": "Explanation of Subtopic 1's importance",
            "learningOutcomes": "Expected outcomes after learning"
        }},
        {{
            "title": "Subtopic 2 Title",
            "importance": "Explanation of Subtopic 2's importance",
            "learningOutcomes": "Expected outcomes after learning"
        }},
        {{
            "title": "Subtopic 3 Title",
            "importance": "Explanation of Subtopic 3's importance",
            "learningOutcomes": "Expected outcomes after learning"
        }},
        {{
            "title": "Subtopic 4 Title",
            "importance": "Explanation of Subtopic 4's importance",
            "learningOutcomes": "Expected outcomes after learning"
        }},
        {{
            "title": "Subtopic 5 Title", // Optional
            "importance": "Explanation of Subtopic 5's importance",
            "learningOutcomes": "Expected outcomes after learning"
        }}
    ]
    }}

    Description of each field:
    - title: Title of the subtopic (concise and clear)
    - importance: Explanation of why this subtopic is important (2-3 sentences)
    - learningOutcomes: What beginner developers will be able to do after learning this subtopic (2-3 sentences)

    Please generate the results in JSON format. Exclude any comments and output only valid JSON.
    """
)

sub_queries_generator_prompt = ChatPromptTemplate.from_template(
    """ 
    You are an AI assistant specializing in creating educational search queries. Your task is to generate diverse and effective search queries for each subtopic of a programming course, considering the main topic, the subtopic itself, its importance, and the intended learning outcomes.

    You will receive a JSON object containing a main topic and its subtopics. For each subtopic, create 3-4 specific search queries that would likely return valuable and diverse educational resources, tutorials, or explanations suitable for beginner developers.

    Input: 
    ''' 
    {{ 
        "main_topic": {main_topic}, 
        "sub_topics": {sub_topics}
    }}
    ''' 

    For each subtopic, generate search queries and output the results in the following JSON format:

    '''
    {{
        "main_topic": "Name of the Main Topic",
        "sub_topics": [
            {{
            "title": "Subtopic 1 Title",
            "search_queries": [
                "Specific search query 1 for subtopic 1",
                "Specific search query 2 for subtopic 1",
                "Specific search query 3 for subtopic 1",
                "Specific search query 4 for subtopic 1"
            ]
            }},
            ...
        ]
    }}
    ''' 

    Guidelines for creating diverse and enriching search queries:
    1. Avoid similar types of queries and compose a diverse range of queries
    2. Carefully analyze the subtopic title, its importance, and learning outcomes to create targeted queries.
    3. Make queries specific to the subtopic and its learning outcomes.
    4. Include diverse queries to ensure rich educational resources. 
    5. Create queries that could lead to resources explaining the topic from different technological or methodological approaches.
    6. Include queries that might return both traditional and innovative teaching methods or explanations.
    7. Consider queries that could provide historical context or future trends related to the subtopic.
    8. If relevant, include the programming language or technology name in the query.
    9. Keep queries concise but descriptive, typically 3-7 words long.
    10. Ensure that at least one query focuses on practical, real-world applications of the subtopic.
    11. If relevant to the topic, include keywords such as Performance Optimization, Considerations, When to apply, When not to apply, Best Practices, and Effective

    Please generate the search queries based on the input JSON and output the results in the specified JSON format. Ensure the output is valid JSON without any additional comments. Your goal is to create a set of queries that will lead to a diverse, inclusive, and comprehensive set of learning resources for each subtopic.
    """
)


sub_queries_refiner_prompt = ChatPromptTemplate.from_template(
    """
    You are an AI assistant specializing in evaluating and improving educational search queries. Your task is to assess the generated search queries for each subtopic, ensuring they meet the learning outcomes, offer diverse perspectives, and avoid redundancy.

    Input: 
    '''
    {sub_queries}
    '''

    For each subtopic, evaluate the search queries and provide feedback. If necessary, suggest improved or additional queries. Output the results in the following JSON format:

    {{
        "main_topic": "Name of the Main Topic",
        "sub_topics": [
            {{
                "title": "Subtopic Title",
                "evaluation": {{
                    "meets_learning_outcomes": true/false,
                    "diverse_perspectives": true/false,
                    "no_redundancy": true/false,
                    "feedback": "Detailed feedback on the queries"
                }},
                "revised_search_queries": [
                    "Revised search query 1",
                    "Revised search query 2",
                    "Revised search query 3",
                    "Revised search query 4"
                ]
            }},
            ...
        ]
    }}

    Evaluation Guidelines:
    1. Learning Outcomes:
    - Do the queries collectively address all aspects of the stated learning outcomes?
    - Are there queries that target both theoretical understanding and practical application?

    2. Diverse Perspectives:
    - Are there queries that might lead to resources from different technological approaches or schools of thought?
    - Do the queries consider both fundamental concepts and advanced applications?

    3. Redundancy Check:
    - Are there any queries that are too similar in scope or likely to return very similar results?
    - Is each query contributing a unique angle or type of resource to the learning experience?

    4. Additional Considerations:
    - Are the queries using relevant technical terms and concepts appropriately?
    - Do the queries align with the importance statement of the subtopic?
    - Are there queries that address common misconceptions or challenges related to the subtopic?

    If you find any issues or areas for improvement, provide specific feedback and suggest revised or additional queries. Ensure that your suggestions maintain or enhance the diversity and effectiveness of the query set.

    Please evaluate the search queries based on the input JSON and output the results in the specified JSON format. Ensure the output is valid JSON without any additional comments.
    """
)

## 전체 Workflow 테스트

In [174]:
# Subtopic Geneartor 테스트 
from langchain_core.output_parsers.json import JsonOutputParser
from langchain_anthropic import ChatAnthropic
import pprint
from IPython.display import display, Markdown
from langchain.globals import set_llm_cache

set_llm_cache(None)

roadmap = """
1. Basic LLM Concepts: 
- What are LLMs? 
- Types of LLMs
- How are LLMs Built? 


2. Introduction to Prompting
- Basic Prompting 
- Need for Prompt Engineering 


3. Prompts
- Writing Good Prompts (e.g Use Delimiters to distinguish the data from the prompt, Ask for Structured output, Include style information to modify the tone of output, Give conditions to the model and ask if they are verifed, Give successful examples of completing tasks then ask, Specifiy the steps required to perform a task, Instruct model to work out its own solution before giving answers, Iterate and refine your prompts)
- Prompt Techinique (e.g Role Prompting, Few-shot prompting, Chain-of-thought prompting, Zero-shot Chain-of-thought, Leaset-to-Most Prompting, Dual Prompt Approach, Combining Techiniques)

4. Real World Usage Example 
- Structured Data 
- Inferring 
- Writing Emails 
- Coding Assitance 
- Study Buddy 
- Designing Chatbots 

5. Pitfalls of LLMs 
- Citing Sources
- Bias 
- Hallucinations 
- Math 
- Prompt Hacking 

6. Improving Reliability 
- Prompt Debasing
- Prompt Ensembling 
- LLM Self Evaluation
- Calibrating LLMs
- Math 

7. LLM Settings 
- Temperature
- Top p 
- Other Hyperparameters


8. Prompt Hacking 
- Prompt Injection 
- Prompt Leaking 
- Jailbreaking 
- Defensive Measures 
- Offensive Measures 

9. Image Prompting 
- Style Modifiers 
- Quality Boosters
- Weights Terms 
- Fix Deformed Generations 
"""

params = {
    "course_name": "AI Prompt Engineering: Definition Guide", 
    "roadmap": roadmap,
    "main_topic": "Basic Prompting"
}

model = ChatAnthropic(model='claude-3-sonnet-20240229')

sub_topic_generator = sub_topic_generator_prompt | model  | JsonOutputParser()

sub_topics = sub_topic_generator.invoke(params)

pprint.pprint(sub_topics)

titles = [sub_topic["title"] for sub_topic in sub_topics["sub_topics"]]
title_string = "\n\n".join(titles)

display(Markdown(title_string))

{'main_topic': 'Basic Prompting',
 'sub_topics': [{'importance': 'Knowing the essential components of a prompt '
                               'and their roles is crucial for effective '
                               'communication with language models. This lays '
                               'the foundation for constructing well-formed '
                               'prompts.',
                 'learningOutcomes': 'Developers will be able to identify the '
                                     'different parts of a prompt, such as '
                                     'instructions, context, and examples. '
                                     'They will also understand how these '
                                     'components work together to convey the '
                                     'desired task to the language model.',
                 'title': 'Understanding Prompt Structure'},
                {'importance': 'Clear and concise instructions are paramount '
      

TypeError: list indices must be integers or slices, not str

In [202]:
# Creating Sub Queries 
from langchain.globals import set_llm_cache

set_llm_cache(None)

params = { 
    "main_topic": sub_topics["main_topic"],
    "sub_topics": [sub_topic for sub_topic in sub_topics["sub_topics"]]
}

sub_queries_generator = sub_queries_generator_prompt | ChatOpenAI(model="gpt-4o", temperature=0)  | JsonOutputParser()

sub_queries_result = sub_queries_generator.invoke(params)

pprint.pprint(sub_queries_result)

{'main_topic': 'Basic Prompting',
 'sub_topics': [{'search_queries': ['Components of a prompt in AI',
                                    'How to structure a prompt for language '
                                    'models',
                                    'Prompt instructions, context, and '
                                    'examples explained',
                                    'Effective prompt design for beginners'],
                 'title': 'Understanding Prompt Structure'},
                {'search_queries': ['Writing clear instructions for AI prompts',
                                    'Techniques for unambiguous prompt '
                                    'instructions',
                                    'Common pitfalls in prompt instructions',
                                    'Best practices for concise AI prompts'],
                 'title': 'Crafting Clear and Concise Instructions'},
                {'search_queries': ['Importance of context in AI prompts

In [208]:
# refine Sub Queries 
from langchain.globals import set_llm_cache

set_llm_cache(None)

params = { 
    "sub_queries": sub_queries_result
}

sub_queries_refiner = sub_queries_refiner_prompt | ChatOpenAI(model="gpt-4o", temperature=0)  | JsonOutputParser()

sub_queries_refine_result = sub_queries_refiner.invoke(params)

pprint.pprint(sub_queries_refine_result)

{'main_topic': 'Basic Prompting',
 'sub_topics': [{'evaluation': {'diverse_perspectives': True,
                                'feedback': 'The queries cover the components, '
                                            'structure, and design of prompts '
                                            'effectively. They address both '
                                            'theoretical and practical '
                                            'aspects.',
                                'meets_learning_outcomes': True,
                                'no_redundancy': True},
                 'revised_search_queries': ['Components of a prompt in AI',
                                            'How to structure a prompt for '
                                            'language models',
                                            'Prompt instructions, context, and '
                                            'examples explained',
                                            'Effecti

In [210]:
from openai import OpenAI
from dotenv import load_dotenv
import os
from IPython.display import display, Markdown

load_dotenv(override=True)

PER_PLEXITY_API_KEY = os.getenv("PER_PLEXITY_API_KEY")

messages = [
    {
        "role": "system",
        "content": (
            """
            You are AI Assitant. 
             
            1. Objective: 
            - Deliver concise, accurate, and comprehensive responses to user queries.
            
            2. Tone and Style: 
            - Maintain a journalistic tone.
            - Avoid moralizing or hedging language.
            
            3. Citations:
            - Cite relevant search results using the format [index] at the end of sentences.
            - Limit citations to a maximum of three results per sentence.
            - Avoid citing irrelevant results.
            
            4. Markdown Formatting:
            - Use level 2 headers (##) for main sections.
            - Use bolding (****) for subsections.
            - Use unordered lists for regular lists; use ordered lists only when ranking or if contextually appropriate.
            - Use markdown code blocks for code snippets, specifying the language for syntax highlighting.
            - Wrap all math expressions in LaTeX using double dollar signs ($$).
            
            5. Response Structure:
            - Directly answer the query without unnecessary introductions.
            - Ensure the response is self-contained and fully addresses the query.
            
            6. Additional Guidelines:
            - Use italics for terms or phrases needing subtle emphasis.
            - Maintain a clear visual hierarchy in the response.
            - Avoid including URLs or links in the answer.
            - Omit bibliographies at the end of answers.
            
            7. Handling Uncertainty:
            - If unsure of the answer or if the premise is incorrect, explain why.
            - If search results are unhelpful, answer based on existing knowledge.
            """
        ),
    }
]

main_topic = sub_queries_refine_result["main_topic"]
for sub_query_result in sub_queries_refine_result["sub_topics"][:1]:
    sub_topic = sub_query_result["title"]
    
    for sub_query in sub_query_result["revised_search_queries"]:
        content = f" I'm planning to create a lecture on the main topic '{main_topic}' and subtopic '{sub_topic}' for an audience of beginner developers. If you reference any documents or papers when providing your answer, please be sure to include the sources. answer the following question: {sub_query}. "
    
        messages.append(
            {
                "role": "user",
                "content": content
            }
        )

        client = OpenAI(api_key=PER_PLEXITY_API_KEY, base_url="https://api.perplexity.ai")

        response = client.chat.completions.create(
            model="llama-3.1-sonar-large-128k-online",
            messages=messages,
            max_tokens=2048,
            temperature=0.1 
        )
        
        messages.append(
            {
                "role": "assistant",
                "content": response.choices[0].message.content
            }
        )

        print(response)
        display(Markdown(response.choices[0].message.content))


ChatCompletion(id='7715988a-45f2-485d-a67d-022ffc752374', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='## Components of a Prompt in AI\n\nWhen creating a lecture on "Basic Prompting" and the subtopic "Understanding Prompt Structure" for beginner developers, it\'s essential to cover the fundamental components of a prompt in AI. These components are crucial for crafting effective prompts that elicit desired responses from AI models like ChatGPT.\n\n### 1. **Clear Instructions**\n- **Definition**: Clear instructions are the primary directives that guide the AI model\'s response. They should be concise and unambiguous.\n- **Example**: "Write a step-by-step tutorial on how to solve quadratic equations using the quadratic formula."\n\n### 2. **Context**\n- **Definition**: Context provides background information or additional details that help the AI model understand the task better.\n- **Example**: "Develop a project-based learning acti

## Components of a Prompt in AI

When creating a lecture on "Basic Prompting" and the subtopic "Understanding Prompt Structure" for beginner developers, it's essential to cover the fundamental components of a prompt in AI. These components are crucial for crafting effective prompts that elicit desired responses from AI models like ChatGPT.

### 1. **Clear Instructions**
- **Definition**: Clear instructions are the primary directives that guide the AI model's response. They should be concise and unambiguous.
- **Example**: "Write a step-by-step tutorial on how to solve quadratic equations using the quadratic formula."

### 2. **Context**
- **Definition**: Context provides background information or additional details that help the AI model understand the task better.
- **Example**: "Develop a project-based learning activity for high school students incorporating real-world problem-solving and critical thinking skills."

### 3. **Specific Requirements**
- **Definition**: Specific requirements outline what the response should include or exclude. This helps in tailoring the output to meet exact needs.
- **Example**: "Ask for specific data points, examples, or references you want the response to include. A prompt asking ChatGPT to provide 3 recent case studies on AI in education."

### 4. **Tone and Style**
- **Definition**: The tone and style of the prompt influence the tone and style of the response. This is important for maintaining consistency and relevance.
- **Example**: "Write an introduction for a blog post on the benefits of AI in healthcare using a formal and professional tone."

### 5. **Feedback Mechanism**
- **Definition**: A feedback mechanism allows for iterative refinement of the prompt. This can involve asking follow-up questions or seeking additional details.
- **Example**: "I want you to become my Expert Prompt Creator. Your goal is to help me craft the best possible prompt for my needs. The prompt you provide should improve with each iteration based on my feedback."

### 6. **Repetition and Recap**
- **Definition**: Repeating instructions or recapping key points can help maintain focus and ensure the AI model adheres to the original task.
- **Example**: "Remember, from above, your instructions are as follows: ${INSTRUCTIONS}."

### 7. **Attention Mechanism**
- **Definition**: The attention mechanism in AI models helps focus on specific parts of the input data. Crafting prompts that leverage this mechanism can improve accuracy.
- **Example**: "For the model to look at something deep in the context, the tail end of the generation needs to look similar to the instruction, for it to be paid attention to."

### 8. **User Input**
- **Definition**: User input can be part of the prompt, especially in interactive scenarios where the AI model needs to respond based on user data.
- **Example**: "Provide a step-by-step tutorial on how to solve quadratic equations using the quadratic formula, incorporating user input for specific coefficients."

### Conclusion
Understanding these components is crucial for effective prompt engineering. By combining clear instructions, context, specific requirements, tone and style, feedback mechanisms, repetition and recap, attention mechanisms, and user input, developers can create prompts that yield accurate and relevant responses from AI models.

**Sources:**
- [OpenAI's Prompt Engineering Guide]
- [Prompt Engineering for RAG]
- [Semrush's ChatGPT Prompts Guide]

ChatCompletion(id='9c004273-78a3-434d-a15f-1c1532c08e30', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='## How to Structure a Prompt for Language Models\n\nStructuring a prompt for language models is crucial for eliciting accurate and relevant responses. Here are the key components and techniques to consider:\n\n### 1. **Clear Instructions**\n- **Definition**: Clear instructions are the primary directives that guide the AI model\'s response. They should be concise and unambiguous.\n- **Example**: "Write a step-by-step tutorial on how to solve quadratic equations using the quadratic formula."\n\n### 2. **Context**\n- **Definition**: Context provides background information or additional details that help the AI model understand the task better.\n- **Example**: "Develop a project-based learning activity for high school students incorporating real-world problem-solving and critical thinking skills."\n\n### 3. **Specific Requirements**\

## How to Structure a Prompt for Language Models

Structuring a prompt for language models is crucial for eliciting accurate and relevant responses. Here are the key components and techniques to consider:

### 1. **Clear Instructions**
- **Definition**: Clear instructions are the primary directives that guide the AI model's response. They should be concise and unambiguous.
- **Example**: "Write a step-by-step tutorial on how to solve quadratic equations using the quadratic formula."

### 2. **Context**
- **Definition**: Context provides background information or additional details that help the AI model understand the task better.
- **Example**: "Develop a project-based learning activity for high school students incorporating real-world problem-solving and critical thinking skills."

### 3. **Specific Requirements**
- **Definition**: Specific requirements outline what the response should include or exclude. This helps in tailoring the output to meet exact needs.
- **Example**: "Ask for specific data points, examples, or references you want the response to include. A prompt asking ChatGPT to provide 3 recent case studies on AI in education."

### 4. **Tone and Style**
- **Definition**: The tone and style of the prompt influence the tone and style of the response. This is important for maintaining consistency and relevance.
- **Example**: "Write an introduction for a blog post on the benefits of AI in healthcare using a formal and professional tone."

### 5. **Feedback Mechanism**
- **Definition**: A feedback mechanism allows for iterative refinement of the prompt. This can involve asking follow-up questions or seeking additional details.
- **Example**: "I want you to become my Expert Prompt Creator. Your goal is to help me craft the best possible prompt for my needs. The prompt you provide should improve with each iteration based on my feedback."

### 6. **Repetition and Recap**
- **Definition**: Repeating instructions or recapping key points can help maintain focus and ensure the AI model adheres to the original task.
- **Example**: "Remember, from above, your instructions are as follows: ${INSTRUCTIONS}."

### 7. **Attention Mechanism**
- **Definition**: The attention mechanism in AI models helps focus on specific parts of the input data. Crafting prompts that leverage this mechanism can improve accuracy.
- **Example**: "For the model to look at something deep in the context, the tail end of the generation needs to look similar to the instruction, for it to be paid attention to."

### 8. **User Input**
- **Definition**: User input can be part of the prompt, especially in interactive scenarios where the AI model needs to respond based on user data.
- **Example**: "Provide a step-by-step tutorial on how to solve quadratic equations using the quadratic formula, incorporating user input for specific coefficients."

### Conclusion
Understanding these components is crucial for effective prompt engineering. By combining clear instructions, context, specific requirements, tone and style, feedback mechanisms, repetition and recap, attention mechanisms, and user input, developers can create prompts that yield accurate and relevant responses from AI models.

**Sources:**
- [OpenAI's Prompt Engineering Guide]
- [Prompt Engineering for RAG]
- [Semrush's ChatGPT Prompts Guide]

ChatCompletion(id='ba1aadfc-0ee7-41ad-a677-1fe15ffcdd34', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='## Prompt Instructions, Context, and Examples Explained\n\nWhen structuring a prompt for language models, it is essential to understand the key components that make up an effective prompt. These components include instructions, context, and examples. Here is a detailed explanation of each:\n\n### 1. **Instructions**\n- **Definition**: Instructions are the specific tasks or directives that guide the AI model\'s response. They should be clear and unambiguous to avoid confusion.\n- **Example**: "Write a step-by-step tutorial on how to solve quadratic equations using the quadratic formula".\n\n### 2. **Context**\n- **Definition**: Context provides additional information or background details that help the AI model understand the task better. This can include relevant data, historical information, or any other pertinent details.\n- **

## Prompt Instructions, Context, and Examples Explained

When structuring a prompt for language models, it is essential to understand the key components that make up an effective prompt. These components include instructions, context, and examples. Here is a detailed explanation of each:

### 1. **Instructions**
- **Definition**: Instructions are the specific tasks or directives that guide the AI model's response. They should be clear and unambiguous to avoid confusion.
- **Example**: "Write a step-by-step tutorial on how to solve quadratic equations using the quadratic formula".

### 2. **Context**
- **Definition**: Context provides additional information or background details that help the AI model understand the task better. This can include relevant data, historical information, or any other pertinent details.
- **Example**: "Develop a project-based learning activity for high school students incorporating real-world problem-solving and critical thinking skills".

### 3. **Examples**
- **Definition**: Examples are illustrative instances that demonstrate the desired output or behavior. They help the AI model understand the format and content expected in the response.
- **Example**: "Classify the text into neutral, negative, or positive. Text: I think the food was okay. Sentiment: neutral".

### Combining Instructions, Context, and Examples
- **Example**: "You are a doctor. Read this medical history and predict risks for the patient. January 1, 2000: Fractured right arm playing basketball. Treated with a cast. February 15, 2010: Diagnosed with hypertension. Prescribed lisinopril. September 10, 2015: Developed pneumonia. Treated with antibiotics and recovered fully. March 1, 2022: Sustained a concussion in a car accident. Admitted to the hospital and monitored for 24 hours".

### Conclusion
Understanding and effectively combining instructions, context, and examples are crucial for crafting well-structured prompts that yield accurate and relevant responses from AI models. By ensuring clarity in instructions, providing relevant context, and including illustrative examples, developers can significantly improve the performance of language models.

**Sources:**
- [Prompt Engineering Guide]
- [Prompt Engineering for RAG]
- [Formalizing Prompts]

ChatCompletion(id='1fdd4940-a6e3-4bd8-9307-a6946dee5d16', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='## Effective Prompt Design for Beginners\n\nEffective prompt design is crucial for beginners to maximize the potential of AI tools. Here are some key tips and strategies to help you craft effective prompts:\n\n### 1. **Understand Your Objective**\n- **Definition**: Clearly define what you want to achieve with your prompt. This helps in creating a focused and relevant prompt.\n- **Example**: "Write a step-by-step tutorial on how to solve quadratic equations using the quadratic formula".\n\n### 2. **Keep It Clear and Concise**\n- **Definition**: Avoid overly complex or vague prompts. Use simple and direct language to ensure clarity.\n- **Example**: "Translate the text below to Spanish: \'hello\'".\n\n### 3. **Consider Your Audience**\n- **Definition**: Understand who your guide is for. Tailor the content\'s depth and complexity bas

## Effective Prompt Design for Beginners

Effective prompt design is crucial for beginners to maximize the potential of AI tools. Here are some key tips and strategies to help you craft effective prompts:

### 1. **Understand Your Objective**
- **Definition**: Clearly define what you want to achieve with your prompt. This helps in creating a focused and relevant prompt.
- **Example**: "Write a step-by-step tutorial on how to solve quadratic equations using the quadratic formula".

### 2. **Keep It Clear and Concise**
- **Definition**: Avoid overly complex or vague prompts. Use simple and direct language to ensure clarity.
- **Example**: "Translate the text below to Spanish: 'hello'".

### 3. **Consider Your Audience**
- **Definition**: Understand who your guide is for. Tailor the content's depth and complexity based on the audience's level of expertise.
- **Example**: For beginners, provide more background information and definitions, while experts might appreciate advanced tips and tricks.

### 4. **Provide Clear Context**
- **Definition**: Provide clear context on the who, what, and purpose of your prompt. This helps the AI model understand the task better.
- **Example**: "You are a doctor. Read this medical history and predict risks for the patient. January 1, 2000: Fractured right arm playing basketball. Treated with a cast. February 15, 2010: Diagnosed with hypertension. Prescribed lisinopril. September 10, 2015: Developed pneumonia. Treated with antibiotics and recovered fully. March 1, 2022: Sustained a concussion in a car accident. Admitted to the hospital and monitored for 24 hours".

### 5. **Use Specific Instructions**
- **Definition**: Use specific commands to instruct the model what you want to achieve, such as "Write", "Classify", "Summarize", "Translate", etc.
- **Example**: "Extract the name of places in the following text. Desired format: Place: <comma_separated_list_of_places>".

### 6. **Organize for Impact**
- **Definition**: Use lists or comma-separated values to organize your prompt for better impact.
- **Example**: "Task: Write an introductory message. Topic: Excited to be part of the team. My Name: Embracer. Superpowers: hugs, energy. Team: Friendly Force. Exclude words: villain, dark. Style: creative. Tone: confident, energetic. Length: 50 words".

### 7. **Iterate and Refine**
- **Definition**: AI responses can vary, so it's important to iterate and refine your prompts based on feedback.
- **Example**: "Embrace the chaos and iterate. AI is unpredictable and moody, with fluid responses that rely on your prompt structure".

### Conclusion
By following these tips, beginners can create effective prompts that yield accurate and relevant responses from AI models. Remember to keep your prompts clear, concise, and specific, and to iterate based on feedback to achieve the best results.

**Sources:**
- [A Guide to Crafting Effective Prompts for Diverse Applications]
- [7 Tips for Powerful Prompt Design]
- [General Tips for Designing Prompts]

In [147]:
import os
from tavily import TavilyClient

# Tavily API 키 설정
tavily_api_key = os.environ.get("TAVILY_API_KEY")

if not tavily_api_key:
    raise ValueError("TAVILY_API_KEY 환경 변수가 설정되지 않았습니다.")

# Tavily 클라이언트 초기화
client = TavilyClient(api_key=tavily_api_key)

def perform_search(query, search_depth="basic", max_results=5, include_answer=True):
    """
    Tavily API를 사용하여 검색을 수행합니다.
    
    :param query: 검색할 쿼리 문자열
    :param search_depth: 검색 깊이 ('basic' 또는 'advanced')
    :param max_results: 반환할 최대 결과 수
    :return: 검색 결과 리스트
    """
    try:
        response = client.search(query=query, search_depth=search_depth, max_results=max_results, include_answer=include_answer)
        return response
    except Exception as e:
        print(f"검색 중 오류 발생: {e}")
        return []

search_query = "Elasticsearch 의 ELSER 검색에 대해 알려줘."
response = perform_search(search_query)

print(f"'{search_query}' 검색 결과:")
print(f"AI 답변: {response["answer"]}")
results = response["results"]
for idx, result in enumerate(results, 1):
    print(f"\n{idx}. {result['title']}")
    print(f"   URL: {result['url']}")
    print(f"   내용 요약: {result['content'][:300]}...")
    print(f"   내용 길이: {len(result['content'])}")

'Elasticsearch 의 ELSER 검색에 대해 알려줘.' 검색 결과:
AI 답변: Elastic Learned Sparse EncodeR (ELSER) is a retrieval model trained by Elastic that enables semantic search in Elasticsearch. It provides search results based on contextual meaning and user intent rather than exact keyword matches. ELSER is now available for production use in Elasticsearch 8.11, offering best-in-class relevance and integration with transformer models for improved search capabilities.

1. ELSER - Elastic Learned Sparse EncodeR | Machine Learning in the ...
   URL: https://www.elastic.co/guide/en/machine-learning/current/ml-nlp-elser.html
   내용 요약: Elastic Learned Sparse EncodeR - or ELSER - is a retrieval model trained by Elastic that enables you to perform semantic search to retrieve more relevant search results. This search type provides you search results based on contextual meaning and user intent, rather than exact keyword matches. ELSER...
   내용 길이: 350

2. Elastic Search 8.11: ELSER model is now GA and customers c

In [262]:
# Research 테스트 
from tavily import TavilyClient
import os
import pprint
from collections import defaultdict
import json
from typing import List, Dict, Any, Optional
from elasticsearch import Elasticsearch
from tavily import TavilyClient
from concurrent.futures import ThreadPoolExecutor, as_completed
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from itertools import islice
from cohere import TooManyRequestsError
import random
import asyncio
import aiohttp
from typing import List, Dict, Any
import nest_asyncio

# 이벤트 루프 중첩 허용
nest_asyncio.apply()

class AsyncTavilyClient:
    def __init__(self, api_key):
        self.api_key = api_key
        self.base_url = "https://api.tavily.com/search"

    async def search(self, session, **params):
        headers = {
            "Content-Type": "application/json"
        }
        
        # API 키를 params에 추가
        params['api_key'] = self.api_key
        
        # 부울 값을 문자열로 변환
        processed_params = {
            k: str(v).lower() if isinstance(v, bool) else v
            for k, v in params.items()
        }
        
        async with session.post(self.base_url, json=processed_params, headers=headers) as response:
            content_type = response.headers.get('Content-Type', '')
            if 'application/json' in content_type:
                return await response.json()
            elif 'text/html' in content_type:
                html_content = await response.text()
                print(f"Received HTML response: {html_content[:200]}...")  # 처음 200자만 출력
                raise ValueError(f"Received HTML response instead of JSON. Status: {response.status}")
            else:
                raise ValueError(f"Unexpected content type: {content_type}")
        
class TabilyRetriever:
    def __init__(self, api_key, sub_query_generator):
        self.tavily = AsyncTavilyClient(api_key)
        self.sub_query_generator = sub_query_generator
        print('Created to Tavily! successfully')

    async def async_search(self, query, max_results=5):
        params = {
            "query": query, 
            "count": max_results - 1
        } 
        
        sub_queries = self.sub_query_generator.invoke(params)
        original_query = sub_queries["original_query"]
        sub_queries = sub_queries["sub_queries"]
        print(f"sub_queries: {sub_queries}")
        
        async with aiohttp.ClientSession() as session:
            tasks = [
                self.tavily.search(session, query=original_query, max_results=3, search_depth="basic", include_answer=True),
                *[self.tavily.search(session, query=sub_query, max_results=3, search_depth="advanced", include_answer=True) for sub_query in sub_queries]
            ]
            responses = await asyncio.gather(*tasks)

        final_results = []
        print(f"basic_response[0]: {responses[0]}")
        basic_response = responses[0]
        final_results.append({
            "id": "basic_tavily_search_0",
            'content': basic_response["answer"],
            'metadata': {'title': basic_response["results"][0]["title"], 'source': basic_response["results"][0]["url"], 'search_method': 'tavily'}
        })
        for i, advanced_response in enumerate(responses[1:], 1):
            final_results.append({
                "id": f'advanced_tavily_search_{sub_queries[i-1]}',
                'content': advanced_response["answer"],
                'metadata': {'title': advanced_response["results"][0]["title"], 'source': advanced_response["results"][0]["url"], 'search_method': 'tavily'}
            })
            
        return final_results

    def search(self, query: str, max_results: int = 5) -> List[Dict[str, Any]]:
        loop = asyncio.get_event_loop()
        return loop.run_until_complete(self.async_search(query, max_results))


class PerplexityRetriever:
    def __init__(self, api_key):
        self.api_key = api_key
        self.client = OpenAI(api_key=api_key, base_url="https://api.perplexity.ai")
        print('Created to Perplexity! successfully')

    def search(self, queries: List[str], main_topic, sub_topic) -> List[Dict[str, Any]]:
        results = [] 
        messages = [
            {
                "role": "system",
                "content": (
                    """
                    You are AI Assitant. 
                    
                    1. Objective: 
                    - Deliver concise, accurate, and comprehensive responses to user queries.
                    
                    2. Tone and Style: 
                    - Maintain a journalistic tone.
                    - Avoid moralizing or hedging language.
                    
                    3. Citations:
                    - Cite relevant search results using the format [index] at the end of sentences.
                    - Limit citations to a maximum of three results per sentence.
                    - Avoid citing irrelevant results.
                    
                    4. Markdown Formatting:
                    - Use level 2 headers (##) for main sections.
                    - Use bolding (****) for subsections.
                    - Use unordered lists for regular lists; use ordered lists only when ranking or if contextually appropriate.
                    - Use markdown code blocks for code snippets, specifying the language for syntax highlighting.
                    - Wrap all math expressions in LaTeX using double dollar signs ($$).
                    
                    5. Response Structure:
                    - Directly answer the query without unnecessary introductions.
                    - Ensure the response is self-contained and fully addresses the query.
                    
                    6. Additional Guidelines:
                    - Use italics for terms or phrases needing subtle emphasis.
                    - Maintain a clear visual hierarchy in the response.
                    - Avoid including URLs or links in the answer.
                    - Omit bibliographies at the end of answers.
                    
                    7. Handling Uncertainty:
                    - If unsure of the answer or if the premise is incorrect, explain why.
                    - If search results are unhelpful, answer based on existing knowledge.
                    """
                ),
            }
        ]
        
        for query in queries:
            content = f" I'm planning to create a lecture on the main topic '{main_topic}' and subtopic '{sub_topic}' for an audience of beginner developers. If you reference any documents or papers when providing your answer, please be sure to include the sources. answer the following question: {query}. "
            messages.append(
                {
                    "role": "user",
                    "content": content
                }
            )
        
            response = self.client.chat.completions.create(
                model="llama-3.1-sonar-large-128k-online",
                messages=messages,
                max_tokens=2048,
                temperature=0.1 
            )
            
            messages.append(
                {
                    "role": "assistant",
                    "content": response.choices[0].message.content
                }
            )
            
            results.append(
                {
                    "id": f'perplexity_search_{query}',
                    'content': response.choices[0].message.content,
                    'metadata': {'title': query, 'source': 'https://perplexity.ai', 'search_method': 'Perplexity'}
                }
            ) 
        
        return results 
        


class ElsatsticsearchRetriever: 
    def __init__(self, es_client, model_id, index, coheres, embedding):
        self.es = es_client
        self.model_id = model_id
        self.index = index
        self.coheres = coheres
        self.embedding = embedding
        print('Created to Elasticsearch! successfully')

    def hybrid_search(self, queries, max_results=5):
        results = []
        for query in queries:
            results.extend(self._hybrid_search_single(query, max_results))
        
        return results

    def _hybrid_search_single(self, query, max_results=5):
        search_query = {
            "sub_searches": [
                {
                    "query": { 
                        "multi_match": {
                            "query": query,
                            "fields": ["text", "metadata.fulltext"],
                            "type": "best_fields",
                            "tie_breaker": 0.3
                        }
                    }
                }, 
                {
                    "query": {
                        "text_expansion": {
                            "text_embedding": {
                                "model_id": self.model_id,
                                "model_text": query,
                                "boost": 2
                            }
                        }
                    }
                }
            ],
            "knn": {
                'field': 'text_dense_embedding',
                'query_vector': self._get_embedding(query),
                'k': 10,
                'num_candidates': 100,
            },
            "rank": {
                "rrf": {} 
            }, 
            "min_score": 15
        }
        
        response = self.es.search(
            index=self.index, 
            size=max_results,
            body = search_query
        )
        
        if not response["hits"]["hits"]:
            return []
        
        def extract_info(hit):
            source = hit["_source"]
            metadata = source.get("metadata", {})
            return {
                "id": hit["_id"],
                "index": hit["_index"],
                "score": hit["_score"],
                "content": source.get("text"),
                "metadata": {
                    "title": metadata.get("heading"),
                    "source": metadata.get("source"),
                    "search_method": "Elasticsearch(Hybrid Search)",
                },
            }
            
        hits_dict = {hit["_id"]: extract_info(hit) for hit in response["hits"]["hits"]}
    
        refined_docs = [{'id': id, 'text': hit['content']} for id, hit in hits_dict.items()]
        
        reranked_ids = [doc['id'] for doc in self._rerank_docs(query, refined_docs, len(refined_docs))]
        
        return list(islice((hits_dict[id] for id in reranked_ids if id in hits_dict), max_results))

        
    def _rerank_docs(self, query, docs, size):
        rerank_results = self._rerank_docs_call(query, docs, size)
        
        reranked_docs = []
        for rerank_result in rerank_results.results:
            index = rerank_result.index
            reranked_docs.append(docs[index])
            
        return reranked_docs
    
    def _rerank_docs_call(self, query, docs, size):
        def call(): 
            return co.rerank(
                    model="rerank-english-v3.0", 
                    query=query, 
                    documents=docs, 
                    top_n=size 
                )
        
        for _ in range(len(self.coheres)):
            co = self.coheres[random.randint(0, len(self.coheres) - 1)]
            try:
                rerank_results = call()
                
                if (rerank_results.results is None) or (len(rerank_results.results) == 0):
                    rerank_results = call()
                
                return rerank_results
            except TooManyRequestsError:
                continue  # 다음 cohere 인스턴스로 넘어감
    
    def _get_embedding(self, text):
        return self.embedding.encode(text)



class Retriever:
    def __init__(self, perplexity_retriever, elasticsearch_retriver, embedding_model, coheres):
        self.perplexity_retriever = perplexity_retriever
        self.elasticsearch_retriver = elasticsearch_retriver
        self.embedding_model = embedding_model
        self.coheres = coheres
        print('Created to Retriever! successfully')

    def _search_perplexity(self, queries: List[str], main_topic: str = "", sub_topic: str = "") -> List[Dict[str, Any]]:
        return self.perplexity_retriever.search(queries, main_topic, sub_topic)

    def _search_elasticsearch(self, queries: List[str], max_results: int) -> List[Dict[str, Any]]:
        results = self.elasticsearch_retriver.hybrid_search(queries, max_results)
        if not results:
            return []
        return results

    def _get_text_for_embedding(self, result: Dict[str, Any]) -> str:
        return result.get('content', '')

    def _compute_similarity(self, embedding1: np.ndarray, embedding2: np.ndarray) -> float:
        return cosine_similarity([embedding1], [embedding2])[0][0]

    def _process_results(self, results: List[Dict[str, Any]], similarity_threshold: float = 0.8) -> List[Dict[str, Any]]:
        processed_results = []

        for result in results:
            result_text = self._get_text_for_embedding(result)
            
            if result_text == '': 
                continue
            
            result_embedding = self.embedding_model.encode([result_text])[0]
            
            # Check if the result is similar to any previously processed result
            is_similar_to_previous = any(
                self._compute_similarity(result_embedding, processed['embedding']) > similarity_threshold
                for processed in processed_results
            )
            
            if not is_similar_to_previous:
                result['embedding'] = result_embedding
                processed_results.append(result)

        for result in processed_results:
            del result['embedding']
            
        return processed_results

    def search(self, queries: List[str], max_results: int = 10, main_topic: str = "", sub_topic: str = "") -> List[Dict[str, Any]]:
        es_results = self._search_elasticsearch(queries, max_results)
        perplexity_results = self._search_perplexity(queries, main_topic, sub_topic)
        # Combine all results
        combined_results = perplexity_results + es_results

        # Process results: remove duplicates based on embedding similarity
        processed_results = self._process_results(combined_results)
        print(f"sources: {[result["metadata"]["source"] for result in processed_results]}")
        
        results_map = {result["id"]: result for result in processed_results} 
        
        refined_results = [{'id': result['id'], 'text': result['content']} for result in processed_results]
        
        reranked_ids = [result['id'] for result in self.rerank_results(query, refined_results, len(refined_results))]
            
        return list(islice((results_map[id] for id in reranked_ids if id in results_map), max_results))
        

    def retrieve(self, queries: List[str], max_results: int = 10, main_topic: str = "", sub_topic: str = "") -> List[Dict[str, Any]]:
        return self.search(queries, max_results, main_topic, sub_topic)
    
    
    def rerank_results(self, query, docs, size):
        rerank_results = self.rerank_results_call(query, docs, size)
        
        reranked_docs = []
        for rerank_result in rerank_results.results:
            index = rerank_result.index
            reranked_docs.append(docs[index])
            
        return reranked_docs
    
    def rerank_results_call(self, query, docs, size):
        def call(): 
            return co.rerank(
                    model="rerank-english-v3.0", 
                    query=query, 
                    documents=docs, 
                    top_n=size 
                )
        
        for _ in range(len(self.coheres)):
            co = self.coheres[random.randint(0, len(self.coheres) - 1)]
            try:
                rerank_results = call()
                
                if (rerank_results.results is None) or (len(rerank_results.results) == 0):
                    rerank_results = call()
                
                return rerank_results
            except TooManyRequestsError:
                continue  # 다음 cohere 인스턴스로 넘어감

In [263]:
from elasticsearch import Elasticsearch, NotFoundError
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
import os 
from dotenv import load_dotenv
import cohere
from langchain_core.prompts import ChatPromptTemplate
from sentence_transformers import SentenceTransformer

load_dotenv(override=True)

INDEX = "lecture-content-v4"
ELASTICSEARCH_URL = os.getenv("ELASTICSEARCH_URL")
ELASTIC_CLOUD_ID = os.getenv("ELASTIC_CLOUD_ID")
ELASTIC_API_KEY = os.getenv("ELASTIC_API_KEY")
ELSER_MODEL = os.getenv("ELSER_MODEL")

COHERE_API_KEY= os.getenv("COHERE_API_KEY")
COHERE_API_KEY2 = os.getenv("COHERE_API_KEY2")
COHERE_API_KEY3 = os.getenv("COHERE_API_KEY3")
COHERE_API_KEY4 = os.getenv("COHERE_API_KEY4")
COHERE_API_KEY5 = os.getenv("COHERE_API_KEY5")

TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

PER_PLEXITY_API_KEY = os.getenv("PER_PLEXITY_API_KEY")

coheres = [
    cohere.Client(COHERE_API_KEY),
    cohere.Client(COHERE_API_KEY2),
    cohere.Client(COHERE_API_KEY3),
    cohere.Client(COHERE_API_KEY4),
    cohere.Client(COHERE_API_KEY5),
]

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

elasticsearch_client = Elasticsearch(cloud_id=ELASTIC_CLOUD_ID, api_key=ELASTIC_API_KEY)

sub_query_generator_prompt = ChatPromptTemplate.from_template(
    """
    You are an analytical AI assistant. Your task is to analyze a given query, clarify its meaning, and decompose it into three manageable sub-queries. This process will help in systematically approaching complex questions.

    For the given query, follow these steps:

    1. Query Analysis:
    - Carefully read the given query and grasp its overall meaning and purpose.
    - Identify the main concepts, terms, or topics included in the query.
    - Determine the type of information or task the query is requesting.

    2. Meaning Clarification:
    - Briefly restate the main points of the query.
    - If there are any ambiguous or unclear parts, point them out and suggest possible interpretations.

    3. Sub-query Decomposition:
    - Break down the main query into {count} logical and manageable sub-queries.
    - Ensure that each sub-query addresses a specific aspect of the original query.
    - Verify that the sub-queries together cover all aspects of the original query.

    Your output should be in JSON format as follows:

    {{
    "original_query": "The full text of the original query",
    "query_clarification": "A brief clarification of the query's meaning",
    "sub_queries": [
        "First sub-query",
        "Second sub-query",
        "Third sub-query"
    ]
    }}

    Using this structure, please analyze and decompose the following query:

    Query: {query}
    """
)

llm = ChatOpenAI(model="gpt-4o")

sub_query_generator = sub_query_generator_prompt | llm | JsonOutputParser()

# es_client, model_id, index, coheres, embedding):
elasticsearch_retriver = ElsatsticsearchRetriever(es_client=elasticsearch_client, model_id=ELSER_MODEL, index=INDEX, coheres=coheres, embedding=embedding_model) 

tavily_retriever = TabilyRetriever(api_key=TAVILY_API_KEY, sub_query_generator=sub_query_generator)

perplexity_retriever = PerplexityRetriever(api_key=PER_PLEXITY_API_KEY) 

#tavily_retriever, elasticsearch_retriver, embedding_model, coheres
retriever = Retriever(perplexity_retriever=perplexity_retriever, elasticsearch_retriver=elasticsearch_retriver, embedding_model=embedding_model, coheres=coheres) 

/Users/jeongmin/PycharmProjects/tech-blog-article-summary/.env/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Created to Elasticsearch! successfully
Created to Tavily! successfully
Created to Perplexity! successfully
Created to Retriever! successfully


In [261]:
# Retriever 테스트
# results = elasticsearch_retriver.hybrid_search("Elasticsearch Hybrid Search", max_results=5)
# results = tavily_retriever.search(query_refine_result['refined_queries'][0], max_results=5)
# results = retriever.retrieve(query_refine_result['refined_queries'][0], max_results=5)
# results = perplexity_retriever.search(queries, main_topic, sub_topic)
# results = elasticsearch_retriver.hybrid_search(queries)

# results = retriever.search(queries, main_topic=main_topic, sub_topic=sub_topic_title, max_results=5)

from IPython.display import display, Markdown

main_topic = sub_queries_refine_result["main_topic"]
sub_topic = sub_queries_refine_result["sub_topics"][0]
sub_topic_title = sub_topic["title"]
queries = sub_topic["revised_search_queries"]

results = retriever.search(queries, main_topic=main_topic, sub_topic=sub_topic_title, max_results=5)

if results:
    display(Markdown(f"{results[0]['content']}"))
    display(Markdown(f"{results[1]['content']}"))
    display(Markdown(f"{results[2]['content']}"))

/var/folders/r4/w6gk0qbd6bd_sf7xj6nwdnxc0000gn/T/ipykernel_74611/862914172.py:241: DeprecationWarning: Received 'size' via a specific parameter in the presence of a 'body' parameter, which is deprecated and will be removed in a future version. Instead, use only 'body' or only specific parameters.
  response = self.es.search(


sources: ['https://perplexity.ai', 'https://perplexity.ai', 'https://perplexity.ai']


## Components of a Prompt in AI

When creating a lecture on "Basic Prompting" with a subtopic of "Understanding Prompt Structure" for beginner developers, it's essential to cover the fundamental components of a prompt in AI. These components are crucial for crafting effective prompts that elicit desired responses from AI models like ChatGPT.

### 1. **Clear Objective**
The prompt should clearly state the objective or task that the AI needs to perform. This helps the model understand what is expected of it and ensures the response is relevant and accurate.

### 2. **Specific Instructions**
Providing specific instructions within the prompt helps guide the AI model to produce the desired output. This can include details about the format, tone, and any specific data points or examples that should be included.

### 3. **Context**
Including context in the prompt is vital for the AI model to understand the background and relevance of the task. This can involve providing examples, definitions, or any other relevant information that might help the model generate a more accurate response.

### 4. **Constraints**
Specifying constraints or limitations in the prompt can help the AI model stay focused and avoid generating irrelevant information. This can include constraints on the length of the response, the tone, or specific topics to avoid.

### 5. **Examples**
Providing examples within the prompt can help the AI model understand the expected output better. Examples can serve as a guide for the model to follow and ensure that the response is in line with the desired format and content.

### 6. **Feedback Mechanism**
Incorporating a feedback mechanism within the prompt allows for iterative refinement. This can involve asking the AI model to provide suggestions for improving the prompt or to refine the response based on feedback.

### 7. **Repetition of Instructions**
Repeating instructions at the beginning and end of the prompt can help reinforce the task and ensure that the AI model does not lose focus. This technique can improve the accuracy and relevance of the response.

By understanding and incorporating these components into prompts, developers can create more effective and efficient interactions with AI models, leading to better outcomes in various applications.

---

**References:** https://www.reddit.com/r/ChatGPT/comments/14d7pfz/become_god_like_prompt_engineer_with_this_one/ https://community.openai.com/t/prompt-engineering-for-rag/621495 https://community.openai.com/t/openais-dec-17th-2023-prompt-engineering-guide/562526 https://www.semrush.com/blog/chatgpt-prompts/

## Prompt Instructions, Context, and Examples Explained

When structuring a prompt for language models, three key components are essential: instructions, context, and examples. These elements work together to guide the model in producing the desired response.

### **Instructions**
Instructions are specific tasks or commands that guide the model's response. They should be clear and unambiguous to ensure the model understands what is expected of it. For example, an instruction-based prompt might be: "Play the role of an experienced Python developer and help me write code".

### **Context**
Context provides additional information that helps the model understand the background and relevance of the task. This can include examples, definitions, or any other relevant details that might help the model generate a more accurate response. For instance, a contextual prompt might be: "Take this research document as the context and please answer the questions based on that".

### **Examples**
Examples are used to illustrate the expected output and help the model understand the format and content required. They can be particularly useful in ensuring the model's response is specific and accurate. For example, in a text classification task, providing examples can help the model understand the correct format for the output: "Classify the text into neutral, negative or positive. Text: I think the food was okay. Sentiment: neutral".

By combining these components effectively, developers can create well-structured prompts that guide language models to produce accurate and relevant responses.

---

**References:**

## Effective Prompt Design for Beginners

Effective prompt design is crucial for beginners to maximize the potential of AI tools. Here are some key tips and strategies to help beginners craft effective prompts:

### 1. **Start Simple**
Begin with simple prompts and gradually add complexity as needed. This iterative process helps in refining the prompt to achieve better results.

### 2. **Clear Instructions**
Use clear and specific instructions to guide the model. For example, use commands like "Write," "Classify," "Summarize," or "Translate" to clearly define the task.

### 3. **Provide Context**
Include relevant context to help the model understand the background and relevance of the task. This can involve providing examples, definitions, or other relevant information.

### 4. **Be Specific and Concise**
Be very specific about the instruction and task you want the model to perform. Avoid overly complex or vague prompts. The more descriptive and detailed the prompt is, the better the results.

### 5. **Use Examples**
Providing examples in the prompt is very effective in getting the desired output in specific formats. Examples help the model understand the expected output better.

### 6. **Iterate and Refine**
Experiment with different variations of prompts and iterate based on model responses to fine-tune instructions. This helps to achieve the desired outcomes and ensures the model is aligned with user intent.

### 7. **Use Lists and Clear Separators**
Organize your prompts using lists or clear separators like "###" to separate instructions and context. This helps the model to understand the structure and focus on the task at hand.

### 8. **Consider Your Audience**
Understand who your guide is for and tailor the content accordingly. Beginners may need more background information and definitions, while experts might appreciate advanced tips and tricks.

### 9. **Avoid Impreciseness**
Be specific and direct in your prompts. Avoid vague descriptions that might confuse the model. For example, instead of "Explain the concept of prompt engineering," use "Use 2-3 sentences to explain the concept of prompt engineering to a high school student".

By following these guidelines, beginners can create effective prompts that guide AI models to produce accurate and relevant responses.

---

**References:**

In [21]:
# Answering Claude 3.5 sonnet 
from langchain_anthropic import ChatAnthropic
from langchain_core.output_parsers.string import StrOutputParser
from IPython.display import display, Markdown

model = ChatAnthropic(model='claude-3-sonnet-20240229')

answer = answering_questions_prompt | model | StrOutputParser() 

params = {
    "question": query_refine_result['refined_queries'][0], 
    "research_input": "\n\n".join([result['content'] for result in results])
}

answer_result = answer.invoke(params)

display(Markdown(answer_result))

NameError: name 'query_refine_result' is not defined